## LiMPIEZA E INTREGACIÓN DE LOS DATOS EN LA VISTA MINABLE
-  Autor: Germán Homero Morán Figueroa
- Descripción: Este notebook permite realizar la limpieza y depuración  del archivo rasta_maiz_con_inferidas, se realizan diferentes transformaciones relacionadas con la textura y rompimiento del suelo. Al final del proceso se obtiene un dataframe que abarca información del suelo.
-  Salida: Vista Minable depurada con información del suelo.

In [1]:
# Librerias y dependencias
# =========================================================================
import pandas as pd
import numpy as np
from unidecode import  unidecode
import plotly.express as px
import re
pd.options.display.max_columns = None

## PROCESAMIENTO DEL ARCHIVO RASTA.CSV

In [2]:
# Lectura del archivos- Origen
# ===================================================================================
rastas = pd.read_csv("../../Data/bronze/rasta_maiz_con_inferidas.csv",encoding='latin-1')
evento_cordoba = pd.read_csv("../../Data/Silver/eventos_cordoba_2017.csv")
eventosFinales = pd.read_csv("../../Data/Silver/eventos_Fertilizations.csv")
rastas.head(5)

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo
0,8,11,1.0,PLANO O LLANO,PLANO,2,"7,63","0,0","31,23","FAr,FAr","PLASTICO,PLASTICO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,SI,0.0,0.0,0.0,SI,0.0,0.0,SI,50.0,NO,GRANULAR,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,NO,NaN,POCO AFECTADAS,NO,NO,NO,NO,BUENO,0,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO
1,9,12,1.0,PLANO O LLANO,PLANO,2,"41,29","4,26","4,26","ArL,FAr","MUY PLASTICO,PLASTICO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,SI,0.0,0.0,0.0,SI,0.0,0.0,SI,56.0,NO,SIN ESTRUCTURA,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,SI,42.0,PLANTAS NORMALES,NO,NO,NO,NO,BUENO,42,"BAJA,BAJA",BUENO,NINGUNO
2,10,13,1.0,PLANO O LLANO,PLANO,2,"40,30","10,18","6,18","FAr,FAr","MUY PLASTICO,PLASTICO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,SI,0.0,0.0,0.0,SI,0.0,0.0,SI,50.0,NO,SIN ESTRUCTURA,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,SI,55.0,PLANTAS NORMALES,NO,NO,NO,NO,MUY BUENO,55,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO
3,12,15,1.0,PLANO O LLANO,PLANO,2,"47,23","18,1","18,1","Ar,A","MUY PLASTICO,MUY PLASTICO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,0.0,0.0,SI,47.0,NO,ATERRONADA,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,SI,47.0,PLANTAS NORMALES,NO,NO,NO,NO,BUENO,47,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO
4,13,16,2.0,ONDULADO,PLANO,4,"21,22,11,13","10,10,10,10","6,18,18,18","FA,FAr,A,FAr","FIRME,FRIABLE,FRIABLE,FRIABLE",6.0,NO TIENE,NaN,PEDREGOSO,SIN ROCAS,PEDREGOSO,SIN ROCAS,SI,2.0,21.0,1.0,SI,2.0,21.0,SI,54.0,NO,MASIVA,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,SI,54.0,PLANTAS NORMALES,NO,NO,NO,NO,SIN COBERTURA,54,"BAJA,BAJA,MEDIA,BAJA",BUENO,NINGUNO


In [3]:
# Se obtienen los IDLotes Unicos para el departamento de cordoba
# ===============================================================
Lista_ID_lotes_cordoba = list(evento_cordoba.ID_LOTE.unique())
print("Numero de lotes unicos del departamento de Cordoba :",len(Lista_ID_lotes_cordoba))

Numero de lotes unicos del departamento de Cordoba : 882


In [4]:
# Se filtra los registros rasta unicamente para el departamento de Cordoba
# =======================================================================================
Lista_ID_lotes_cordoba
rastasCordoba = rastas[rastas.ID_LOTE.isin(Lista_ID_lotes_cordoba)].reset_index(drop=True)
print(rastasCordoba.shape)
rastasCordoba.head() 

(810, 48)


,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo
0,40,42,2.0,PLANO O LLANO,PLANO,3,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,NaN,NaN,SI,31.0,NO,GRANULAR,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,NO,NaN,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO
1,43,43,1.0,PLANO O LLANO,PLANO,3,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,NO,NaN,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO
2,44,44,1.0,PLANO O LLANO,PLANO,3,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO
3,45,45,2.0,PLANO O LLANO,PLANO,3,"16,39,29","4,18,31","9,44,34","FAr,FrL,FrL","BLANDO,BLANDO,BLANDO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,22.0,1.5,SI,11.0,NO,GRANULAR,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,NO,NaN,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,11,"BAJA,NA,NA",LENTO A MUY LENTO,NINGUNO
4,46,46,2.0,PLANO O LLANO,PLANO,3,"33,6,47","8,32,35","16,34,38","FAr,FAr,FrL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,19.0,1.7,SI,6.0,NO,GRANULAR,NO,NO,NO HAY,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,NO,NaN,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,6,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO


In [5]:
# Verificación de la cantidad de Lotes Unicos
# ============================================
len(rastasCordoba.ID_LOTE.unique())

804

In [6]:
# En este caso no se elimina ningun registro dado que analizando la información no son registros duplicados
rastasCordoba.ID_LOTE.value_counts().head(10)


2937    2
3675    2
3016    2
479     2
670     2
717     2
3642    1
3643    1
3644    1
3667    1
Name: ID_LOTE, dtype: int64

In [7]:
# Eliminamos las filas duplicadas del dataframe
# No existen ninguna fila duplicada..
rastasCordoba[rastasCordoba.duplicated()]

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo


In [8]:
# Comprobacion de que realmente se filtro por los lotes correctos
rastasCordoba[rastasCordoba.ID_LOTE==2937]

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo
457,2937,2811,1.0,PLANO O LLANO,PLANO,2,"26,34","10,49","7,44","Ar,FAr","DURO,BLANDO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,NaN,NaN,SI,27.0,NO,GRANULAR,NO,NO,POCO MARCADAS,LA MAÑANA Y LA TARDE,POCO MARCADAS,NO HAY,NO,SI,17.0,PLANTAS NORMALES,NO,NO,SI,NO,BUENO,27,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO
458,2937,2811,2.0,PLANO O LLANO,PLANO,2,"27,33","10,49","2,31","Ar,F","DURO,BLANDO",5.5,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,12.0,4.0,SI,28.0,NO,GRANULAR,NO,NO,POCO MARCADAS,LA MAÑANA Y LA TARDE,NO HAY,NO HAY,NO,SI,16.0,PLANTAS NORMALES,NO,NO,SI,NO,BUENO,16,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO


*Nota*: Se observa que a pesar de tener IDs de lotes iguales, los registros son diferentes es decir difieren en algunas columnas como: Espesor y Textura.

In [9]:
# Verificamos qeu efectivamente no existe ninguna Fila duplicada
rastasCordoba.duplicated().value_counts()

False    810
dtype: int64

In [10]:
# Verificación de la distribución de algunas columnas
# ==================================================
rastasCordoba.SITIO_EXPUESTO_SOL_RASTA.value_counts()


LA MAÑANA Y LA TARDE    806
LA MAÑANA                 3
LA TARDE                  1
Name: SITIO_EXPUESTO_SOL_RASTA, dtype: int64

In [11]:
# Manejo de caracteres especiales [Elimnacion de las Tildes]
# ===========================================================================
rastasCordoba.TERRENO_CIRCUN_RASTA = rastasCordoba.TERRENO_CIRCUN_RASTA.apply(lambda x: "ONDULADO Y MONTANOSO" if x=="ONDULADO Y MONTAÑOSO" else x)
rastasCordoba.POSICION_PERFIL_RASTA = rastasCordoba.POSICION_PERFIL_RASTA.apply(lambda x: "PIE DE UNA ELEVACION" if x=="PIE DE UNA ELEVACIÓN" else x)
rastasCordoba.POSICION_PERFIL_RASTA = rastasCordoba.POSICION_PERFIL_RASTA.apply(lambda x: "LADERA CONCAVA" if x=="LADERA CÓNCAVA" else x)
rastasCordoba.SITIO_EXPUESTO_SOL_RASTA = rastasCordoba.SITIO_EXPUESTO_SOL_RASTA.apply(lambda x: "LA MANANA Y LA TARDE" if x =="LA MAÑANA Y LA TARDE" else x)
rastasCordoba.SITIO_EXPUESTO_SOL_RASTA = rastasCordoba.SITIO_EXPUESTO_SOL_RASTA.apply(lambda x: "LA MANANA" if x =="LA MAÑANA" else x)


In [12]:
#  Verificación de variables y formato.
# ====================================================
rastasCordoba.POSICION_PERFIL_RASTA.value_counts()

PLANO                     780
PLANO CON ONDULACIONES     18
LADERA CONVEXA              5
LADERA PLANA                3
PIE DE UNA ELEVACION        2
LADERA CONCAVA              2
Name: POSICION_PERFIL_RASTA, dtype: int64

In [13]:
rastasCordoba.OBSERVA_RAICES_VIVAS_RASTA.value_counts()

SI    778
NO     32
Name: OBSERVA_RAICES_VIVAS_RASTA, dtype: int64

In [14]:
rastasCordoba.PROFUND_RAICES_VIVAS_RASTA

0       NaN
1       NaN
2      30.0
3       NaN
4       NaN
       ... 
805    20.0
806    27.0
807    23.0
808    31.0
809    22.0
Name: PROFUND_RAICES_VIVAS_RASTA, Length: 810, dtype: float64

In [15]:
rastasCordoba[["OBSERVA_RAICES_VIVAS_RASTA", "PROFUND_RAICES_VIVAS_RASTA"]]

,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA
0,NO,NaN
1,NO,NaN
2,SI,30.0
3,NO,NaN
4,NO,NaN
...,...,...
805,SI,20.0
806,SI,27.0
807,SI,23.0
808,SI,31.0


In [16]:
''' 
    Al aplicar el filtro anterior [OBSERVA_RAICES_VIVAS_RASTA, PROFUND_RAICES_VIVAS_RASTA]
    cuando en los registros la variable OBSERVA_RAICES_VIVAS_RASTA [NO], tienen asignado NAN
    Sin emabargo cuando no hay presencia de RAICES RASTA no debe existir ninguna profundidad
    por lo tanto se asigna el valor de (-1) a estos valores.

'''

rastasCordoba.PROFUND_RAICES_VIVAS_RASTA[rastasCordoba.OBSERVA_RAICES_VIVAS_RASTA=='NO'] = -1
rastasCordoba.PROFUND_CAP_ENDURE_RASTA[rastasCordoba.CAP_ENDURE_RASTA=='NO']= -1
rastasCordoba.ESPESOR_CAP_ENDURE_RASTA[rastasCordoba.CAP_ENDURE_RASTA == 'NO'] = -1
rastasCordoba.PROFUND_MOTEADOS_RASTA[rastasCordoba.CAP_ENDURE_RASTA == 'NO'] = -1


<ipython-input-16-77c252e16066>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rastasCordoba.PROFUND_RAICES_VIVAS_RASTA[rastasCordoba.OBSERVA_RAICES_VIVAS_RASTA=='NO'] = -1
<ipython-input-16-77c252e16066>:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rastasCordoba.PROFUND_CAP_ENDURE_RASTA[rastasCordoba.CAP_ENDURE_RASTA=='NO']= -1
<ipython-input-16-77c252e16066>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rastasCordoba.ESPES

In [17]:
# Verificación de las Transformaciones
# =========================================================================
rastasCordoba.PROFUND_MOTEADOS_RASTA[rastasCordoba.CAP_ENDURE_RASTA == 'NO']

0     -1.0
14    -1.0
20    -1.0
22    -1.0
23    -1.0
      ... 
804   -1.0
805   -1.0
806   -1.0
807   -1.0
808   -1.0
Name: PROFUND_MOTEADOS_RASTA, Length: 707, dtype: float64

In [18]:
# Verificación del tratameinto sobre estas variables
# ===================================================================
rastasCordoba.PROFUND_MOTEADOS_RASTA.unique()

array([-1., 30., 14., 11.,  6., 20., 37., 55., 50., 15., 23., 64., 26.,
       nan, 13., 35., 21., 24., 12., 25., 22., 31., 19., 18., 28., 27.])

###  Tratamiento de datos del suelo mediante la guia Rasta

En este apartado se realiza la transformación y procesamiento de la variable Textura
asociada a cada evento del cultivo. Se extrae los porcentajes de de presencia de c/u 
de las texturas.

In [19]:
# Se crea un nuevo dataframe donde se almacenan las diferentes texturas del suelo de acuerdo  a la Guia Rasta
# ===============================================================================================================
tablaTexturas = pd.DataFrame(["A","Ar","ArA","ArL","FrL","L","F","ArF","FAr","FA","AF"], columns=['textura'])
tablaTexturas

,textura
0,A
1,Ar
2,ArA
3,ArL
4,FrL
5,L
6,F
7,ArF
8,FAr
9,FA


In [20]:
# Procedimiento de calcular y almacenar en una lista el porcentaje 
# de cada posible textua al interior del cajon que se realiza en la guia rasta.
capas = rastasCordoba[['NO_CAPAS_RASTA','ESPESOR','TEXTURA']]
capas

,NO_CAPAS_RASTA,ESPESOR,TEXTURA
0,3,"22,9,50","FAr,FrL,FA"
1,3,"20,14,36","FAr,FAr,ArL"
2,3,"26,14,38","FAr,FAr,AF"
3,3,"16,39,29","FAr,FrL,FrL"
4,3,"33,6,47","FAr,FAr,FrL"
...,...,...,...
805,3,"20,18,22","FAr,FAr,Ar"
806,2,"25,35","FAr,Ar"
807,2,"28,32","ArL,Ar"
808,2,"38,22","Ar,Ar"


In [21]:
''' 
El numero de capas que se encuentra en la tierra 
varian de acuerdo a las capas encontradas al abrir el cajon
La presencia de capas varia de un terreno a otro
'''
capas.TEXTURA.value_counts()

FAr,FAr        123
Ar,Ar           98
Ar,FAr          78
FAr,F           57
Ar,F            47
              ... 
ArL,FrL,L        1
FAr,L,FrL        1
FAr,FAr,ArA      1
ArA,FA           1
FrL,ArL          1
Name: TEXTURA, Length: 91, dtype: int64

In [22]:
def divisionCadenas(x):
    return x.split(",")

def divisionEspesor(x):
    lista1 = x.split(",")
    dE= [int(x) for x in lista1]
    return np.array(dE)

def ProfundidadTotal(x):
    a = [int(b) for b in x]
    return sum(a)

In [23]:
# Se dividen la TEXTURA Y ESPESOR en diferentes listas
# ================================================================================
rastasCordoba["DivisionCadenas"] = rastasCordoba.TEXTURA.apply(divisionCadenas)
rastasCordoba["DivisionEspesor"]= rastasCordoba.ESPESOR.apply(divisionEspesor)


In [24]:
# Se Verifica la división del Espesor
# ==================================================
rastasCordoba.DivisionCadenas

0       [FAr, FrL, FA]
1      [FAr, FAr, ArL]
2       [FAr, FAr, AF]
3      [FAr, FrL, FrL]
4      [FAr, FAr, FrL]
            ...       
805     [FAr, FAr, Ar]
806          [FAr, Ar]
807          [ArL, Ar]
808           [Ar, Ar]
809          [FAr, Ar]
Name: DivisionCadenas, Length: 810, dtype: object

In [25]:
rastasCordoba.DivisionEspesor[0]

array([22,  9, 50])

In [27]:
# Se obtiene la profundidad total que es la suma de c/u de las capas de los espresores.
# ======================================================================================
rastasCordoba["ProfundidadTotal"]= rastasCordoba.DivisionEspesor.apply(ProfundidadTotal)

In [28]:
rastasCordoba.ProfundidadTotal

0      81
1      70
2      78
3      84
4      86
       ..
805    60
806    60
807    60
808    60
809    60
Name: ProfundidadTotal, Length: 810, dtype: int64

In [29]:
# Verificación de las transformaciones realizadas
# ===============================================
rastasCordoba.head(3)

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,DivisionCadenas,DivisionEspesor,ProfundidadTotal
0,40,42,2.0,PLANO O LLANO,PLANO,3,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FA]","[22, 9, 50]",81
1,43,43,1.0,PLANO O LLANO,PLANO,3,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, ArL]","[20, 14, 36]",70
2,44,44,1.0,PLANO O LLANO,PLANO,3,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, AF]","[26, 14, 38]",78


In [30]:
len(rastasCordoba.ID_LOTE.unique())

804

In [31]:
rastasCordoba.DivisionCadenas[44]

['Ar', 'Ar']

In [32]:
''' 
Se obtiene el porcentaje de Textura asociado a c/d Espresar
'''
def porcentajeTextura(Espesor):
    ListaPorcentajeTextura = []
    s = sum(Espesor)
    #print(s)

    for i in  Espesor:
        r3= (i * 100)/s
        ListaPorcentajeTextura.append(round(r3,2))
    return ListaPorcentajeTextura

In [33]:

'''
    Función para convertir el dataset en formato estandar

'''

def DatasetTransformStandar(df):
    print(len(df))
    # Build dictionary
    for i in range (0,len(df)):
        dicc = {
            'textura':df.loc[i].DivisionCadenas,
            'esp':df.loc[i].DivisionEspesor,
            'porc':df.loc[i].PorcentajeTextura,
            'ID_LOTE':df.loc[i].ID_LOTE
        }

        NewDfm = pd.DataFrame(data=dicc)
        NewDf = NewDfm.groupby(by=["textura","ID_LOTE"]).sum().reset_index(drop=False)
        tablaPorcetajesTexturas = tablaTexturas.set_index('textura').join(NewDf.set_index('textura')).reset_index(drop=False).sort_values(by='porc', ascending=False)
        if i > 0:
            FinalDf = pd.concat([FinalDf,tablaPorcetajesTexturas], ignore_index=True)
        else:
            FinalDf = tablaPorcetajesTexturas


    return FinalDf


In [34]:
# Obtengo los porcetajes de c/u  de los registros asociados a c/u de los valores de textura
# ==========================================================================================
rastasCordoba["PorcentajeTextura"] = rastasCordoba.DivisionEspesor.apply(porcentajeTextura)


In [35]:
rastasCordoba.head(5)

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,DivisionCadenas,DivisionEspesor,ProfundidadTotal,PorcentajeTextura
0,40,42,2.0,PLANO O LLANO,PLANO,3,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FA]","[22, 9, 50]",81,"[27.16, 11.11, 61.73]"
1,43,43,1.0,PLANO O LLANO,PLANO,3,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, ArL]","[20, 14, 36]",70,"[28.57, 20.0, 51.43]"
2,44,44,1.0,PLANO O LLANO,PLANO,3,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, AF]","[26, 14, 38]",78,"[33.33, 17.95, 48.72]"
3,45,45,2.0,PLANO O LLANO,PLANO,3,"16,39,29","4,18,31","9,44,34","FAr,FrL,FrL","BLANDO,BLANDO,BLANDO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,22.0,1.5,SI,11.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,11,"BAJA,NA,NA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FrL]","[16, 39, 29]",84,"[19.05, 46.43, 34.52]"
4,46,46,2.0,PLANO O LLANO,PLANO,3,"33,6,47","8,32,35","16,34,38","FAr,FAr,FrL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,19.0,1.7,SI,6.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,6,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, FrL]","[33, 6, 47]",86,"[38.37, 6.98, 54.65]"


In [36]:
# Genero una copia del dataframe rastas cordoba para el mergue con otros registros
# =================================================================================
rastasCordobaCopy = rastasCordoba.copy()

In [37]:
rastasCordobaCopy.shape

(810, 52)

In [38]:
# Proceso para eleminar los duplicados
# ==============================================================================
rastasCordobaCopy.ID_LOTE.value_counts().head(10)

2937    2
3675    2
3016    2
479     2
670     2
717     2
3642    1
3643    1
3644    1
3667    1
Name: ID_LOTE, dtype: int64

In [39]:
rastasCordoba[rastasCordoba.ID_LOTE==717]

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,DivisionCadenas,DivisionEspesor,ProfundidadTotal,PorcentajeTextura
66,717,712,2.0,PLANO O LLANO,PLANO,2,"28,42","10,23","18,36","Ar,Ar","DURO,DURO",5.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,SI,MASIVA,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,33.0,PLANTAS NORMALES,NO,NO,NO,SI,REGULAR,33,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[Ar, Ar]","[28, 42]",70,"[40.0, 60.0]"
67,717,712,2.0,PLANO O LLANO,PLANO,2,"28,42","10,23","19,36","Ar,Ar","DURO,DURO",5.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,MASIVA,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,29.0,PLANTAS NORMALES,NO,NO,NO,SI,REGULAR,29,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[Ar, Ar]","[28, 42]",70,"[40.0, 60.0]"


In [40]:
rastasCordobaCopy

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,DivisionCadenas,DivisionEspesor,ProfundidadTotal,PorcentajeTextura
0,40,42,2.0,PLANO O LLANO,PLANO,3,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FA]","[22, 9, 50]",81,"[27.16, 11.11, 61.73]"
1,43,43,1.0,PLANO O LLANO,PLANO,3,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, ArL]","[20, 14, 36]",70,"[28.57, 20.0, 51.43]"
2,44,44,1.0,PLANO O LLANO,PLANO,3,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, AF]","[26, 14, 38]",78,"[33.33, 17.95, 48.72]"
3,45,45,2.0,PLANO O LLANO,PLANO,3,"16,39,29","4,18,31","9,44,34","FAr,FrL,FrL","BLANDO,BLANDO,BLANDO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,22.0,1.5,SI,11.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,11,"BAJA,NA,NA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FrL]","[16, 39, 29]",84,"[19.05, 46.43, 34.52]"
4,46,46,2.0,PLANO O LLANO,PLANO,3,"33,6,47","8,32,35","16,34,38","FAr,FAr,FrL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,19.0,1.7,SI,6.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,6,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, FrL]","[33, 6, 47]",86,"[38.37, 6.98, 54.65]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,4379,4615,3.0,PLANO O LLANO,PLANO,3,"20,18,22","25,9,17","20,6,29","FAr,FAr,Ar","FRIABLE,FRIABLE,FIRME",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,NO,-1.0,NO,ATERRONADA,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,20.0,PLANTAS NORMALES,NO,NO,SI,NO,BUENO,60,"MEDIA,NA,NA",BUENO,LENTO,"[FAr, FAr, Ar]","[20, 18, 22]",60,"[33.33, 30.0, 36.67]"
806,4380,4616,4.0,PLANO O LLANO,PLANO,2,"25,35","27,22","18,14","FAr,Ar","FRIABLE,FIRME",5.5,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,NO,-1.0,NO,ATERRONADA,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,27.0,POCO

In [41]:
# Se elimnian 6 registros del datframe original, porque ID_LOTE esta duplicado
# Indice a eliminar [457,546,485,22,37,66]
rastasCordobaCopy.drop([457,546,485,22,37,66], axis=0, inplace=True)


In [42]:
rastasCordobaCopy = rastasCordobaCopy.reset_index(drop=True)
rastasCordobaCopy.head(5)

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,DivisionCadenas,DivisionEspesor,ProfundidadTotal,PorcentajeTextura
0,40,42,2.0,PLANO O LLANO,PLANO,3,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FA]","[22, 9, 50]",81,"[27.16, 11.11, 61.73]"
1,43,43,1.0,PLANO O LLANO,PLANO,3,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, ArL]","[20, 14, 36]",70,"[28.57, 20.0, 51.43]"
2,44,44,1.0,PLANO O LLANO,PLANO,3,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, AF]","[26, 14, 38]",78,"[33.33, 17.95, 48.72]"
3,45,45,2.0,PLANO O LLANO,PLANO,3,"16,39,29","4,18,31","9,44,34","FAr,FrL,FrL","BLANDO,BLANDO,BLANDO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,22.0,1.5,SI,11.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,11,"BAJA,NA,NA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FrL]","[16, 39, 29]",84,"[19.05, 46.43, 34.52]"
4,46,46,2.0,PLANO O LLANO,PLANO,3,"33,6,47","8,32,35","16,34,38","FAr,FAr,FrL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,19.0,1.7,SI,6.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,6,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, FrL]","[33, 6, 47]",86,"[38.37, 6.98, 54.65]"


In [43]:
# Se crea el dataframe texturas para realizar la conversion de los datos
# Se crea un nuevo dataframe donde se almacenan las diferentes texturas del suelo
tablaTexturas = pd.DataFrame(["A","Ar","ArA","ArL","FrL","L","F","ArF","FAr","FA","AF"], columns=['textura'])
tablaTexturas

,textura
0,A
1,Ar
2,ArA
3,ArL
4,FrL
5,L
6,F
7,ArF
8,FAr
9,FA


In [44]:
# Filtramos el dataset rastas_cordoba para mandarle unicamente los campos de interes
# - textura
# - Espesro
# - Porcentaje
# - id

dfEstandar = rastasCordobaCopy[['DivisionCadenas','DivisionEspesor','PorcentajeTextura','ID_LOTE']]
dfEstandar

,DivisionCadenas,DivisionEspesor,PorcentajeTextura,ID_LOTE
0,"[FAr, FrL, FA]","[22, 9, 50]","[27.16, 11.11, 61.73]",40
1,"[FAr, FAr, ArL]","[20, 14, 36]","[28.57, 20.0, 51.43]",43
2,"[FAr, FAr, AF]","[26, 14, 38]","[33.33, 17.95, 48.72]",44
3,"[FAr, FrL, FrL]","[16, 39, 29]","[19.05, 46.43, 34.52]",45
4,"[FAr, FAr, FrL]","[33, 6, 47]","[38.37, 6.98, 54.65]",46
...,...,...,...,...
799,"[FAr, FAr, Ar]","[20, 18, 22]","[33.33, 30.0, 36.67]",4379
800,"[FAr, Ar]","[25, 35]","[41.67, 58.33]",4380
801,"[ArL, Ar]","[28, 32]","[46.67, 53.33]",4381
802,"[Ar, Ar]","[38, 22]","[63.33, 36.67]",4382


In [45]:
# Aplicamos la funcion DatasetTransformStandar
# =======================================================
porcentajes_textura = DatasetTransformStandar(dfEstandar)


804


In [46]:
porcentajes_textura

,textura,ID_LOTE,esp,porc
0,FA,40.0,50.0,61.73
1,FAr,40.0,22.0,27.16
2,FrL,40.0,9.0,11.11
3,A,NaN,NaN,NaN
4,Ar,NaN,NaN,NaN
...,...,...,...,...
8839,L,NaN,NaN,NaN
8840,F,NaN,NaN,NaN
8841,ArF,NaN,NaN,NaN
8842,FA,NaN,NaN,NaN


In [47]:
porcentajes_textura.ID_LOTE

0       40.0
1       40.0
2       40.0
3        NaN
4        NaN
        ... 
8839     NaN
8840     NaN
8841     NaN
8842     NaN
8843     NaN
Name: ID_LOTE, Length: 8844, dtype: float64

In [48]:
# Completamos con 0 aquellos campos que no tienen asociado la capa especifica
# ===========================================================================
porcentajes_textura.porc = porcentajes_textura.porc.fillna(0)

In [49]:
porcentajes_textura.ID_LOTE = porcentajes_textura.ID_LOTE.fillna(method="ffill")

# Se guarda la informacion para posterio analisis
porcentajes_textura.to_excel("../../Data/Silver/Rasta/porcentajes_textura.xlsx")


In [50]:
# Filtros de c/u de las texturas
porcentajes_textura_A = porcentajes_textura[porcentajes_textura.textura == 'A'][['ID_LOTE','porc']]
porcentajes_textura_Ar = porcentajes_textura[porcentajes_textura.textura == 'Ar'][['ID_LOTE','porc']]
porcentajes_textura_ArA = porcentajes_textura[porcentajes_textura.textura == 'ArA'][['ID_LOTE','porc']]
porcentajes_textura_ArL = porcentajes_textura[porcentajes_textura.textura == 'ArL'][['ID_LOTE','porc']]
porcentajes_textura_FrL = porcentajes_textura[porcentajes_textura.textura == 'FrL'][['ID_LOTE','porc']]
porcentajes_textura_L = porcentajes_textura[porcentajes_textura.textura == 'L'][['ID_LOTE','porc']]
porcentajes_textura_F = porcentajes_textura[porcentajes_textura.textura == 'F'][['ID_LOTE','porc']]
porcentajes_textura_ArF = porcentajes_textura[porcentajes_textura.textura == 'ArF'][['ID_LOTE','porc']]
porcentajes_textura_FAr = porcentajes_textura[porcentajes_textura.textura == 'FAr'][['ID_LOTE','porc']]
porcentajes_textura_FA = porcentajes_textura[porcentajes_textura.textura == 'FA'][['ID_LOTE','porc']]
porcentajes_textura_AF = porcentajes_textura[porcentajes_textura.textura == 'AF'][['ID_LOTE','porc']]




In [554]:
# se cambio el nombre del ID a la llave del cruze
'''
porcentajes_textura_A.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_Ar.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_ArA.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_ArL.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_FrL.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_L.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_F.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_ArF.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_FAr.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_FA.rename(columns ={'id':'ID_LOTE'},inplace = True)
porcentajes_textura_AF.rename(columns ={'id':'ID_LOTE'},inplace = True)

# Guardamos los porcentajes para las texturas Ar
#porcentajes_textura_Ar.to_excel("../Archivos Generados/porcentajes_textura_Ar.xlsx")'''

'\nporcentajes_textura_A.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_Ar.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_ArA.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_ArL.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_FrL.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_L.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_F.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_ArF.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_FAr.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_FA.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\nporcentajes_textura_AF.rename(columns ={\'id\':\'ID_LOTE\'},inplace = True)\n\n# Guardamos los porcentajes para las texturas Ar\n#porcentajes_textura_Ar.to_excel("../Archivos Generados/porcentajes_textura_Ar.xlsx")'

In [51]:
porcentajes_textura_A

,ID_LOTE,porc
3,40.0,0.0
13,43.0,0.0
24,44.0,0.0
35,45.0,0.0
46,46.0,0.0
...,...,...
8791,4379.0,0.0
8802,4380.0,0.0
8813,4381.0,0.0
8823,4382.0,0.0


In [52]:
# Se va adicionando cada textura con su respectivo porcentaje de presencia al datset original
# =============================================================================================


# Textura A
# ==============
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_A,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_A'},inplace=True)


# Textura Ar
# =============
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_Ar,how='inner')
rastasCordobaCopy.rename(columns={'porc':'Porc_Ar'},inplace=True)

# Textura ArA
# ============
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_ArA,how='inner')
rastasCordobaCopy.rename(columns={'porc':'Porc_ArA'},inplace=True)


# Textura ArL
# =============
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_ArL,how='inner')
rastasCordobaCopy.rename(columns={'porc':'Porc_ArL'},inplace=True)


# Textura FrL
# =============
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_FrL,how='inner')
rastasCordobaCopy.rename(columns={'porc':'Porc_FrL'},inplace=True)


# Textura L
# ===========
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_L,how='inner')
rastasCordobaCopy.rename(columns={'porc':'Porc_L'},inplace=True)

# Textura F
# ===========
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_F,how='inner')
rastasCordobaCopy.rename(columns={'porc':'Porc_F'},inplace=True)

# Textura ArF
# ===========
rastasCordobaCopy= pd.merge(rastasCordobaCopy,porcentajes_textura_ArF,how='inner')
rastasCordobaCopy.rename(columns={'porc':'Porc_ArF'},inplace=True)

# Textura FAr
# ===========
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_FAr,how='inner')
#rastasCordobaCopy.rename(columns={'porc':'porc_FAr'},inplace=True)
rastasCordobaCopy.rename(columns={'porc':'porc.x'},inplace=True)

# Textura FA
# ===========
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_FA,how='inner')
#rastasCordobaCopy.rename(columns={'porc':'porc_FA'},inplace=True)
rastasCordobaCopy.rename(columns={'porc':'porc.y'},inplace=True)

# Textura AF
# ===========
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_textura_AF,how='inner')
rastasCordobaCopy.rename(columns={'porc':'Porc_AF'},inplace=True)


In [53]:
# verificación de las transformaciones aplicadas a c/d registro
# ================================================================
rastasCordobaCopy.head(5)

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,DivisionCadenas,DivisionEspesor,ProfundidadTotal,PorcentajeTextura,Porc_A,Porc_Ar,Porc_ArA,Porc_ArL,Porc_FrL,Porc_L,Porc_F,Porc_ArF,porc.x,porc.y,Porc_AF
0,40,42,2.0,PLANO O LLANO,PLANO,3,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FA]","[22, 9, 50]",81,"[27.16, 11.11, 61.73]",0.0,0.0,0.0,0.00,11.11,0.0,0.0,0.0,27.16,61.73,0.00
1,43,43,1.0,PLANO O LLANO,PLANO,3,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, ArL]","[20, 14, 36]",70,"[28.57, 20.0, 51.43]",0.0,0.0,0.0,51.43,0.00,0.0,0.0,0.0,48.57,0.00,0.00
2,44,44,1.0,PLANO O LLANO,PLANO,3,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, AF]","[26, 14, 38]",78,"[33.33, 17.95, 48.72]",0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,51.28,0.00,48.72
3,45,45,2.0,PLANO O LLANO,PLANO,3,"16,39,29","4,18,31","9,44,34","FAr,FrL,FrL","BLANDO,BLANDO,BLANDO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,22.0,1.5,SI,11.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,11,"BAJA,NA,NA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FrL]","[16, 39, 29]",84,"[19.05, 46.43, 34.52]",0.0,0.0,0.0,0.00,80.95,0.0,0.0,0.0,19.05,0.00,0.00
4,46,46,2.0,PLANO O LLANO,PLANO,3,"33,6,47","8,32,35","16,34,38","FAr,FAr,FrL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,19.0,1.7,SI,6.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,6,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, FrL]","[33, 6, 47]",86,"[38.37, 6.98, 54.65]",0.0,0.0,0.0,0.00,54.65,0.0,0.0,0.0,45.35,0.00,0.00


In [54]:
print("Longitud: ", rastasCordobaCopy.shape)

Longitud:  (804, 63)


### Tratamiento de la  variable resitencia al rompimiento

De manera analoga a la veriable textura se realiza el mismo procedimiento. 

In [55]:
rastasCordoba.RESIST_ROMPIMIENTO[0]

'BLANDO,BLANDO,BLANDO'

In [56]:
#  PROCESAMIENTO VARIABLE RESISTENCIA AL ROMPIMIENTO
# =========================================================================

'''
Se crea una tabla donde se crean todas los tipos de resistencia
asociados al suelo para c/u de las capas

'''
tablaResistencias = pd.DataFrame(["BLANDO","DURO","EXTREMADAMENTE DURO","FRIABLE","FIRME","EXTREMADAMENTE FIRME","PLASTICO","MUY PLASTICO"], columns=['resistencia'])
tablaResistencias

,resistencia
0,BLANDO
1,DURO
2,EXTREMADAMENTE DURO
3,FRIABLE
4,FIRME
5,EXTREMADAMENTE FIRME
6,PLASTICO
7,MUY PLASTICO


In [57]:

# Función para convertir el dataframe de forma normal a estandar
#  Variable (Resistencia al Rompimiento)
# ===================================================================
def DatasetTransformStandarResistencias(df):
    print(len(df))
    for i in range (0,len(df)):
        dicc = {
            'resistencia':df.loc[i].RESIST_ROMPIMIENTO.split(","),
            'acm':df.loc[i].DivisionEspesor,
            'porc':df.loc[i].PorcentajeTextura,
            'ID_LOTE':df.loc[i].ID_LOTE
        }
        NewDfm = pd.DataFrame(data=dicc)
        NewDf = NewDfm.groupby(by=["resistencia","ID_LOTE"]).sum().reset_index(drop=False)

        tablaPorcetajesResistencias = tablaResistencias.set_index('resistencia').join(NewDf.set_index('resistencia')).reset_index(drop=False).sort_values(by='porc', ascending=False)
        if i > 0:
            FinalDf = pd.concat([FinalDf,tablaPorcetajesResistencias], ignore_index=True)
        else:
            FinalDf = tablaPorcetajesResistencias


    return FinalDf

In [58]:
# Verificación de las variables a separar
# ===============================================
rastasCordobaCopy.RESIST_ROMPIMIENTO[0].split(",")

['BLANDO', 'BLANDO', 'BLANDO']

In [59]:
# Se genera DF unicamente con las variables resistencia al rompimiento
# Se realiza el mismo procedimiento utilizado para las texturas.
# ===========================================================================
dfEstandarResistencias = rastasCordobaCopy[['RESIST_ROMPIMIENTO','DivisionEspesor','PorcentajeTextura','ID_LOTE']]
dfEstandarResistencias

,RESIST_ROMPIMIENTO,DivisionEspesor,PorcentajeTextura,ID_LOTE
0,"BLANDO,BLANDO,BLANDO","[22, 9, 50]","[27.16, 11.11, 61.73]",40
1,"BLANDO,BLANDO,BLANDO","[20, 14, 36]","[28.57, 20.0, 51.43]",43
2,"BLANDO,BLANDO,BLANDO","[26, 14, 38]","[33.33, 17.95, 48.72]",44
3,"BLANDO,BLANDO,BLANDO","[16, 39, 29]","[19.05, 46.43, 34.52]",45
4,"BLANDO,BLANDO,BLANDO","[33, 6, 47]","[38.37, 6.98, 54.65]",46
...,...,...,...,...
799,"FRIABLE,FRIABLE,FIRME","[20, 18, 22]","[33.33, 30.0, 36.67]",4379
800,"FRIABLE,FIRME","[25, 35]","[41.67, 58.33]",4380
801,"FRIABLE,FIRME","[28, 32]","[46.67, 53.33]",4381
802,"FIRME,FIRME","[38, 22]","[63.33, 36.67]",4382


In [60]:
# Exportamos las columnas resistencia al rompimiento para posterior analisis
# ==========================================================================
dfEstandarResistencias.to_excel("../../Data/Silver/Rasta/res_rompimiento.xlsx")

In [61]:
# Se obtienen los porcentajes de resistencia
# ================================================================================
PorcentajesResistencia = DatasetTransformStandarResistencias(dfEstandarResistencias)

804


In [62]:
PorcentajesResistencia

,resistencia,ID_LOTE,acm,porc
0,BLANDO,40.0,81.0,100.0
1,DURO,NaN,NaN,NaN
2,EXTREMADAMENTE DURO,NaN,NaN,NaN
3,FRIABLE,NaN,NaN,NaN
4,FIRME,NaN,NaN,NaN
...,...,...,...,...
6427,EXTREMADAMENTE DURO,NaN,NaN,NaN
6428,FRIABLE,NaN,NaN,NaN
6429,EXTREMADAMENTE FIRME,NaN,NaN,NaN
6430,PLASTICO,NaN,NaN,NaN


In [63]:
# Se realiza el mismo procemiento utilizado para la GUIA rasta (TEXTURA)
# ==================================================================================
PorcentajesResistencia.porc = PorcentajesResistencia.porc.fillna(0)
PorcentajesResistencia.ID_LOTE = PorcentajesResistencia.ID_LOTE.fillna(method="ffill")

# Se guarda los datos en el archivo excel
PorcentajesResistencia.to_excel("../../Data/Silver/Rasta/Porcentajes_Resistencias.xlsx")

In [64]:
# Se realiza el mismo  procedimiento de las texturas del suelo para cada una de las porciones
# Filtros de c/u de las resistencias
# =============================================================================================


porcentajes_res_BLANDO = PorcentajesResistencia[PorcentajesResistencia.resistencia == 'BLANDO'][['ID_LOTE','porc']]
porcentajes_res_DURO = PorcentajesResistencia[PorcentajesResistencia.resistencia == 'DURO'][['ID_LOTE','porc']]
porcentajes_res_EXTREMADAMENTE_DURO = PorcentajesResistencia[PorcentajesResistencia.resistencia == 'EXTREMADAMENTE DURO'][['ID_LOTE','porc']]
porcentajes_res_FRIABLE = PorcentajesResistencia[PorcentajesResistencia.resistencia == 'FRIABLE'][['ID_LOTE','porc']]
porcentajes_res_FIRME = PorcentajesResistencia[PorcentajesResistencia.resistencia == 'FIRME'][['ID_LOTE','porc']]
porcentajes_res_EXTREMADAMENTE_FIRME = PorcentajesResistencia[PorcentajesResistencia.resistencia == 'EXTREMADAMENTE FIRME'][['ID_LOTE','porc']]
porcentajes_res_PLASTICO = PorcentajesResistencia[PorcentajesResistencia.resistencia == 'PLASTICO'][['ID_LOTE','porc']]
porcentajes_res_MUY_PLASTICO = PorcentajesResistencia[PorcentajesResistencia.resistencia == 'MUY PLASTICO'][['ID_LOTE','porc']]

In [65]:
# Finalmente se realiza el mergue de c/u de los archivos con el DF final
# ==============================================================================================

# Porcion Blando
# =========================
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_res_BLANDO,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_BLANDO'},inplace=True)


# Porc DURO
# =========================
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_res_DURO ,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_DURO'},inplace=True)


# Porc EXTREMADAMENTE  DURO
# ===========================
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_res_EXTREMADAMENTE_DURO ,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_EXT_DURO'},inplace=True)


# Porc FRIABLE
# ==========================
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_res_FRIABLE,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_FRIABLE'},inplace=True)


# Porc FIRME
# =========================
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_res_FIRME,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_FIRME'},inplace=True)

# Porc Ext FIRME
# ========================
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_res_EXTREMADAMENTE_FIRME,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_EXT_FIRME'},inplace=True)


# PORC PLASTICO
# =========================
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_res_PLASTICO,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_PLASTICO'},inplace=True)

# Por MUY PLASTICO
# =======================
rastasCordobaCopy = pd.merge(rastasCordobaCopy,porcentajes_res_MUY_PLASTICO,how='inner',on=["ID_LOTE"])
rastasCordobaCopy.rename(columns={'porc':'Porc_MUY_PLASTICO'},inplace=True)

In [66]:
rastasCordobaCopy

,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,DivisionCadenas,DivisionEspesor,ProfundidadTotal,PorcentajeTextura,Porc_A,Porc_Ar,Porc_ArA,Porc_ArL,Porc_FrL,Porc_L,Porc_F,Porc_ArF,porc.x,porc.y,Porc_AF,Porc_BLANDO,Porc_DURO,Porc_EXT_DURO,Porc_FRIABLE,Porc_FIRME,Porc_EXT_FIRME,Porc_PLASTICO,Porc_MUY_PLASTICO
0,40,42,2.0,PLANO O LLANO,PLANO,3,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FA]","[22, 9, 50]",81,"[27.16, 11.11, 61.73]",0.0,0.00,0.0,0.00,11.11,0.0,0.0,0.0,27.16,61.73,0.00,100.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0
1,43,43,1.0,PLANO O LLANO,PLANO,3,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, ArL]","[20, 14, 36]",70,"[28.57, 20.0, 51.43]",0.0,0.00,0.0,51.43,0.00,0.0,0.0,0.0,48.57,0.00,0.00,100.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0
2,44,44,1.0,PLANO O LLANO,PLANO,3,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, AF]","[26, 14, 38]",78,"[33.33, 17.95, 48.72]",0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0,51.28,0.00,48.72,100.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0
3,45,45,2.0,PLANO O LLANO,PLANO,3,"16,39,29","4,18,31","9,44,34","FAr,FrL,FrL","BLANDO,BLANDO,BLANDO",6.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,22.0,1.5,SI,11.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,11,"BAJA,NA,NA",LENTO A MUY LENTO,NINGUNO,"[FAr, FrL, FrL]","[16, 39, 29]",84,"[19.05, 46.43, 34.52]",0.0,0.00,0.0,0.00,80.95,0.0,0.0,0.0,19.05,0.00,0.00,100.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0
4,46,46,2.0,PLANO O LLANO,PLANO,3,"33,6,47","8,32,35","16,34,38","FAr,FAr,FrL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,19.0,1.7,SI,6.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,6,"BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,"[FAr, FAr, FrL]","[33, 6, 47]",86,"[38.37, 6.98, 54.65]",0.0,0.00,0.0,0.00,54.65,0.0,0.0,0.0,45.35,0.00,0.00,100.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [67]:
# Verificación de las columnas
# ========================================

# Se conservan columnas necesarias
# =======================================
rastasCordobaCopy.columns

Index(['ID_LOTE', 'ID_FINCA', 'PENDIENTE_RASTA', 'TERRENO_CIRCUN_RASTA',
       'POSICION_PERFIL_RASTA', 'NO_CAPAS_RASTA', 'ESPESOR', 'COLOR_SECO',
       'COLOR_HUMEDO', 'TEXTURA', 'RESIST_ROMPIMIENTO', 'PH_RASTA',
       'CARBONATOS_RASTA', 'PROFUNDIDAD_CARBONATOS', 'PEDREG_SUPERF_PIEDRAS',
       'PEDREG_SUPERF_ROCAS', 'PEDREG_PERFIL_PIEDRAS', 'PEDREG_PERFIL_ROCAS',
       'HOR_PEDREG_ROCOSO_RASTA', 'HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA',
       'HOR_PEDREG_ROCOSO_ESPESOR_RASTA',
       'PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.', 'CAP_ENDURE_RASTA',
       'PROFUND_CAP_ENDURE_RASTA', 'ESPESOR_CAP_ENDURE_RASTA',
       'MOTEADOS_RASTA', 'PROFUND_MOTEADOS_RASTA', 'MOTEADOS_MAS70cm._RASTA',
       'ESTRUCTURA_RASTA', 'OBSERVA_EROSION_RASTA', 'OBSERVA_MOHO_RASTA',
       'OBSERVA_COSTRAS_DURAS_RASTA', 'SITIO_EXPUESTO_SOL_RASTA',
       'OBSERVA_COSTRAS_BLANCAS_RASTA', 'OBSERVA_COSTAS_NEGRAS_RASTA',
       'REGION_SECA_ARIDA_RASTA', 'OBSERVA_RAICES_VIVAS_RASTA',
       'PROFUND_RAICES_VI

In [68]:
# Columnas Elimninar: Son el resultado de operaciónes entre variables
# =======================================================================================================
columunasElimianrAfterProc = ['DivisionCadenas','DivisionEspesor', 'ProfundidadTotal','PorcentajeTextura']
rastasCordobaCopy = rastasCordobaCopy.drop(columunasElimianrAfterProc, axis=1)
rastasCordobaCopy.head(3)


,ID_LOTE,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,Porc_A,Porc_Ar,Porc_ArA,Porc_ArL,Porc_FrL,Porc_L,Porc_F,Porc_ArF,porc.x,porc.y,Porc_AF,Porc_BLANDO,Porc_DURO,Porc_EXT_DURO,Porc_FRIABLE,Porc_FIRME,Porc_EXT_FIRME,Porc_PLASTICO,Porc_MUY_PLASTICO
0,40,42,2.0,PLANO O LLANO,PLANO,3,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,11.11,0.0,0.0,0.0,27.16,61.73,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,43,43,1.0,PLANO O LLANO,PLANO,3,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,51.43,0.00,0.0,0.0,0.0,48.57,0.00,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,44,44,1.0,PLANO O LLANO,PLANO,3,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,51.28,0.00,48.72,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
# Se realiza merge con eventos finales
# ================================================================================================

# Hasta el momento eventos finales continen features de controles y fertilizaciones.
# ==============================================================================================
eventosFinales.head(5)

,ID_EVENTO,ID_LOTE,TIPO_SIEMBRA,SEM_TRATADAS,MATERIAL_GENETICO,CULT_ANT,DRENAJE,METODO_COSECHA,ALMACENAMIENTO_FINCA,AREA,RDT_AJUSTADO,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Antes_Siem,ContEnfQui_Siem_Eme,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Antes_Siem,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,ContMalQui_Antes_Siem,ContMalQui_Siem_Emer,ContMalQui_Emer_Flor,ContMalQui_Flor_Cose,ContPlaQui_Antes_Siem,ContPlaQui_Siem_Emer,ContPlaQui_Emer_Flor,ContPlaQui_Flor_Cose,TotN_Antes_Siem,TotN_Siem_Emer,TotN_Emer_Flor,TotP_Antes_Siem,TotP_Siem_Emer,TotP_Emer_Flor,TotK_Antes_Siem,TotK_Siem_Emer,TotK_Emer_Flor,FerOrg_Antes_Siem,FerOrg_Siem_Emer,FerOrg_Emer_Flor,FerQui_Antes_Siem,FerQui_Siem_Emer,FerQui_Emer_Flor
0,53,40,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,4767.441860,5.0,63.0,68.0,60000,13,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1
1,54,43,Mecanizado,SI,DK 234,Maiz,SI,Manual,NO,1.0,4651.162791,5.0,64.0,63.0,60000,15,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,92.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,2
2,56,44,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,5180.232558,5.0,59.0,66.0,60000,12,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1
3,57,45,Mecanizado,NO,Otro,Algodón,SI,Manual,NO,1.0,4897.674419,5.0,64.0,59.0,60000,12,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1
4,273,46,Mecanizado,NO,Otro,Algodón,SI,Manual,NO,1.0,5302.325581,5.0,63.0,60.0,60000,16,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1


In [70]:
# Dimensiones de los Unificar los 2 dataframes
# =========================================================================
print("Dimension evantos Cordoba Finales (controls+Fertilizacion): ",eventosFinales.shape)
print("Dimensiones del procesamiento de la guia rasta: ",rastasCordobaCopy.shape)

Dimension evantos Cordoba Finales (controls+Fertilizacion):  (882, 47)
Dimensiones del procesamiento de la guia rasta:  (804, 67)


In [71]:
# Se realiza el merge de eventosFinales con rastasCordoba que contiene info del suelo
# =====================================================================================
eventosFinalesv1  = pd.merge(eventosFinales,rastasCordobaCopy,how='left',on=["ID_LOTE"])
eventosFinalesv1

,ID_EVENTO,ID_LOTE,TIPO_SIEMBRA,SEM_TRATADAS,MATERIAL_GENETICO,CULT_ANT,DRENAJE,METODO_COSECHA,ALMACENAMIENTO_FINCA,AREA,RDT_AJUSTADO,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Antes_Siem,ContEnfQui_Siem_Eme,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Antes_Siem,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,ContMalQui_Antes_Siem,ContMalQui_Siem_Emer,ContMalQui_Emer_Flor,ContMalQui_Flor_Cose,ContPlaQui_Antes_Siem,ContPlaQui_Siem_Emer,ContPlaQui_Emer_Flor,ContPlaQui_Flor_Cose,TotN_Antes_Siem,TotN_Siem_Emer,TotN_Emer_Flor,TotP_Antes_Siem,TotP_Siem_Emer,TotP_Emer_Flor,TotK_Antes_Siem,TotK_Siem_Emer,TotK_Emer_Flor,FerOrg_Antes_Siem,FerOrg_Siem_Emer,FerOrg_Emer_Flor,FerQui_Antes_Siem,FerQui_Siem_Emer,FerQui_Emer_Flor,ID_FINCA,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,ESPESOR,COLOR_SECO,COLOR_HUMEDO,TEXTURA,RESIST_ROMPIMIENTO,PH_RASTA,CARBONATOS_RASTA,PROFUNDIDAD_CARBONATOS,PEDREG_SUPERF_PIEDRAS,PEDREG_SUPERF_ROCAS,PEDREG_PERFIL_PIEDRAS,PEDREG_PERFIL_ROCAS,HOR_PEDREG_ROCOSO_RASTA,HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA,HOR_PEDREG_ROCOSO_ESPESOR_RASTA,PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,materia_organica,d.interno,drenaje_externo,Porc_A,Porc_Ar,Porc_ArA,Porc_ArL,Porc_FrL,Porc_L,Porc_F,Porc_ArF,porc.x,porc.y,Porc_AF,Porc_BLANDO,Porc_DURO,Porc_EXT_DURO,Porc_FRIABLE,Porc_FIRME,Porc_EXT_FIRME,Porc_PLASTICO,Porc_MUY_PLASTICO
0,53,40,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,4767.441860,5.0,63.0,68.0,60000,13,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1,42.0,2.0,PLANO O LLANO,PLANO,3.0,"22,9,50","7,10,32","16,26,49","FAr,FrL,FA","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31.0,"BAJA,NA,MEDIA",LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,11.11,0.0,0.0,0.0,27.16,61.73,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,54,43,Mecanizado,SI,DK 234,Maiz,SI,Manual,NO,1.0,4651.162791,5.0,64.0,63.0,60000,15,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,92.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,2,43.0,1.0,PLANO O LLANO,PLANO,3.0,"20,14,36","5,18,31","7,26,32","FAr,FAr,ArL","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20.0,"BAJA,BAJA,BAJA",LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,51.43,0.00,0.0,0.0,0.0,48.57,0.00,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,56,44,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,5180.232558,5.0,59.0,66.0,60000,12,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1,44.0,1.0,PLANO O LLANO,PLANO,3.0,"26,14,38","5,32,18","6,26,26","FAr,FAr,AF","BLANDO,BLANDO,BLANDO",7.0,NO TIENE,NaN,SIN PIEDRAS,SIN ROCAS,SIN PIEDRAS,SIN ROCAS,NO,NaN,NaN,NaN,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30.0,"BAJA,BAJA,MEDIA",LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,51.28,0.00,48.72,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,57,45,Mecanizado,NO,Otro,Algodón,SI,Manual,NO,1.0,4897.674419,5.0,64.0,59.0,60000,12,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,

In [72]:
''' 
Finalmente Se eliminan las siguientes columnas por tener asociados los siguientes caractersiticas:
- Identificadores
- Columnas utilizadas para la generación de otras columnas derivadas
- Columnas con gran cantida de datos faltantes
- Se eliminan las siguientes columnas por no tener variabilidad (Mayor numero registros para 1 Categoria)
'''

# Se definen las columnas a eliminar en la guia rasta.
# =====================================================
borarEnRasta = ["ID_FINCA","CARBONATOS_RASTA","PROFUNDIDAD_CARBONATOS","PEDREG_SUPERF_PIEDRAS","PEDREG_SUPERF_ROCAS","HOR_PEDREG_ROCOSO_RASTA","HOR_PEDREG_ROCOSO_PROFUNDIDAD_RASTA","HOR_PEDREG_ROCOSO_ESPESOR_RASTA","PROFUND_PRIMERAS_ROCAS_PIEDRAS_RASTA..CM.",
                   "ESPESOR","COLOR_SECO","COLOR_HUMEDO","TEXTURA","RESIST_ROMPIMIENTO","materia_organica","PEDREG_PERFIL_PIEDRAS"]


In [73]:
# Finalmente se eliminan las columnas Rasta
# =========================================================
dataframeFinal = eventosFinalesv1.drop(borarEnRasta, axis=1)
print("Dimenciones de la matriz final: ",dataframeFinal.shape)
dataframeFinal.head(3)

Dimenciones de la matriz final:  (882, 97)


,ID_EVENTO,ID_LOTE,TIPO_SIEMBRA,SEM_TRATADAS,MATERIAL_GENETICO,CULT_ANT,DRENAJE,METODO_COSECHA,ALMACENAMIENTO_FINCA,AREA,RDT_AJUSTADO,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Antes_Siem,ContEnfQui_Siem_Eme,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Antes_Siem,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,ContMalQui_Antes_Siem,ContMalQui_Siem_Emer,ContMalQui_Emer_Flor,ContMalQui_Flor_Cose,ContPlaQui_Antes_Siem,ContPlaQui_Siem_Emer,ContPlaQui_Emer_Flor,ContPlaQui_Flor_Cose,TotN_Antes_Siem,TotN_Siem_Emer,TotN_Emer_Flor,TotP_Antes_Siem,TotP_Siem_Emer,TotP_Emer_Flor,TotK_Antes_Siem,TotK_Siem_Emer,TotK_Emer_Flor,FerOrg_Antes_Siem,FerOrg_Siem_Emer,FerOrg_Emer_Flor,FerQui_Antes_Siem,FerQui_Siem_Emer,FerQui_Emer_Flor,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,PH_RASTA,PEDREG_PERFIL_ROCAS,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,d.interno,drenaje_externo,Porc_A,Porc_Ar,Porc_ArA,Porc_ArL,Porc_FrL,Porc_L,Porc_F,Porc_ArF,porc.x,porc.y,Porc_AF,Porc_BLANDO,Porc_DURO,Porc_EXT_DURO,Porc_FRIABLE,Porc_FIRME,Porc_EXT_FIRME,Porc_PLASTICO,Porc_MUY_PLASTICO
0,53,40,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,4767.441860,5.0,63.0,68.0,60000,13,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1,2.0,PLANO O LLANO,PLANO,3.0,7.0,SIN ROCAS,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31.0,LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,11.11,0.0,0.0,0.0,27.16,61.73,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,54,43,Mecanizado,SI,DK 234,Maiz,SI,Manual,NO,1.0,4651.162791,5.0,64.0,63.0,60000,15,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,92.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,2,1.0,PLANO O LLANO,PLANO,3.0,7.0,SIN ROCAS,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20.0,LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,51.43,0.00,0.0,0.0,0.0,48.57,0.00,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,56,44,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,5180.232558,5.0,59.0,66.0,60000,12,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1,1.0,PLANO O LLANO,PLANO,3.0,7.0,SIN ROCAS,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30.0,LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,51.28,0.00,48.72,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [74]:
# Eliminamos los registros de eventos sin registros asociados a la guia rasta es decir no contienen INFO suelo
# ===========================================================================================================

''' 
Verificación de registros:  La información del suelo es relevante para el estudio por tal motivo se verifica
la completitud de las siguientes columnas
    - PENDIENTE_RASTA
    - PH_RASTA
    - ESTRUCTURA_RASTA

En caso  de que lso registros no contengan esta información se eliminan.
'''
dataframeFinal[(dataframeFinal.PENDIENTE_RASTA.isnull()== True) & (dataframeFinal.PH_RASTA.isnull()== True) & (dataframeFinal.ESTRUCTURA_RASTA.isnull()== True)][["PENDIENTE_RASTA","PH_RASTA","ESTRUCTURA_RASTA"]]

,PENDIENTE_RASTA,PH_RASTA,ESTRUCTURA_RASTA
35,NaN,NaN,NaN
39,NaN,NaN,NaN
65,NaN,NaN,NaN
293,NaN,NaN,NaN
409,NaN,NaN,NaN
...,...,...,...
615,NaN,NaN,NaN
696,NaN,NaN,NaN
735,NaN,NaN,NaN
747,NaN,NaN,NaN


In [75]:

# Finalmente se obtiene la matriz final con los datos Faltantes
# =======================================================================
matrizFinal = dataframeFinal[(dataframeFinal.PENDIENTE_RASTA.isnull()== False) & (dataframeFinal.PH_RASTA.isnull()== False) & (dataframeFinal.ESTRUCTURA_RASTA.isnull()== False)].reset_index(drop=True)
print("Longitud Matriz Final: ", matrizFinal.shape)
matrizFinal.head(5)

Longitud Matriz Final:  (804, 97)


,ID_EVENTO,ID_LOTE,TIPO_SIEMBRA,SEM_TRATADAS,MATERIAL_GENETICO,CULT_ANT,DRENAJE,METODO_COSECHA,ALMACENAMIENTO_FINCA,AREA,RDT_AJUSTADO,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Antes_Siem,ContEnfQui_Siem_Eme,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Antes_Siem,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,ContMalQui_Antes_Siem,ContMalQui_Siem_Emer,ContMalQui_Emer_Flor,ContMalQui_Flor_Cose,ContPlaQui_Antes_Siem,ContPlaQui_Siem_Emer,ContPlaQui_Emer_Flor,ContPlaQui_Flor_Cose,TotN_Antes_Siem,TotN_Siem_Emer,TotN_Emer_Flor,TotP_Antes_Siem,TotP_Siem_Emer,TotP_Emer_Flor,TotK_Antes_Siem,TotK_Siem_Emer,TotK_Emer_Flor,FerOrg_Antes_Siem,FerOrg_Siem_Emer,FerOrg_Emer_Flor,FerQui_Antes_Siem,FerQui_Siem_Emer,FerQui_Emer_Flor,PENDIENTE_RASTA,TERRENO_CIRCUN_RASTA,POSICION_PERFIL_RASTA,NO_CAPAS_RASTA,PH_RASTA,PEDREG_PERFIL_ROCAS,CAP_ENDURE_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,MOTEADOS_RASTA,PROFUND_MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,ESTRUCTURA_RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_COSTRAS_DURAS_RASTA,SITIO_EXPUESTO_SOL_RASTA,OBSERVA_COSTRAS_BLANCAS_RASTA,OBSERVA_COSTAS_NEGRAS_RASTA,REGION_SECA_ARIDA_RASTA,OBSERVA_RAICES_VIVAS_RASTA,PROFUND_RAICES_VIVAS_RASTA,OBSERVA_PLANTAS_PEQUENAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,RECUBRIMIENTO_VEGETAL__SUELO_RASTA,prof_efectiva,d.interno,drenaje_externo,Porc_A,Porc_Ar,Porc_ArA,Porc_ArL,Porc_FrL,Porc_L,Porc_F,Porc_ArF,porc.x,porc.y,Porc_AF,Porc_BLANDO,Porc_DURO,Porc_EXT_DURO,Porc_FRIABLE,Porc_FIRME,Porc_EXT_FIRME,Porc_PLASTICO,Porc_MUY_PLASTICO
0,53,40,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,4767.441860,5.0,63.0,68.0,60000,13,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1,2.0,PLANO O LLANO,PLANO,3.0,7.0,SIN ROCAS,NO,-1.0,-1.0,SI,-1.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,NO,NO,SI,NO,MUY BUENO,31.0,LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,11.11,0.0,0.0,0.0,27.16,61.73,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,54,43,Mecanizado,SI,DK 234,Maiz,SI,Manual,NO,1.0,4651.162791,5.0,64.0,63.0,60000,15,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,92.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,2,1.0,PLANO O LLANO,PLANO,3.0,7.0,SIN ROCAS,SI,20.0,2.0,SI,30.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,20.0,LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,51.43,0.00,0.0,0.0,0.0,48.57,0.00,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,56,44,Mecanizado,NO,PIONEER 30F32,Algodón,SI,Manual,NO,1.0,5180.232558,5.0,59.0,66.0,60000,12,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1,1.0,PLANO O LLANO,PLANO,3.0,7.0,SIN ROCAS,SI,18.0,2.0,SI,14.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,SI,30.0,PLANTAS NORMALES,SI,NO,SI,NO,BUENO,30.0,LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,51.28,0.00,48.72,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,57,45,Mecanizado,NO,Otro,Algodón,SI,Manual,NO,1.0,4897.674419,5.0,64.0,59.0,60000,12,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1,2.0,PLANO O LLANO,PLANO,3.0,6.0,SIN ROCAS,SI,22.0,1.5,SI,11.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,11.0,LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,80.95,0.0,0.0,0.0,19.05,0.00,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,273,46,Mecanizado,NO,Otro,Algodón,SI,Manual,NO,1.0,5302.325581,5.0,63.0,60.0,60000,16,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0.0,0.0,46.0,0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,1,2.0,PLANO O LLANO,PLANO,3.0,7.0,SIN ROCAS,SI,19.0,1.7,SI,6.0,NO,GRANULAR,NO,NO,NO HAY,LA MANANA Y LA TARDE,NO HAY,NO HAY,NO,NO,-1.0,PLANTAS NORMALES,SI,NO,NO,NO,BUENO,6.0,LENTO A MUY LENTO,NINGUNO,0.0,0.0,0.0,0.00,54.65,0.0,0.0,0.0,45.35,0.00,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [76]:
# Finalmeente Guardamos los datos, con toda la información en la matriz final.
# Matriz Final es la vista minable que contienen información (Genaral, controles,Fertilizaciones, suelo)
# Se almacena esta matriz en Oro Oro para futuros analisis
# ============================================================================

matrizFinal.to_csv("../../Data/Gold/datasetFinal_soil.csv",index=False)